# Rental Price Prediction Pipeline

## 1.Import Necessary Libraries

In [0]:
import xgboost
print(xgboost.__version__)

3.4.0


In [0]:
pip install xgboost


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


## 2.Import Dataset

In [0]:


file_path = "/Workspace/Users/rohith07.cse@gmail.com/Machine Learning/Rental_Data_cleaned.csv"

Rental_prediction_pipeline = pd.read_csv(file_path)

Rental_prediction_pipeline.head()


,listing_id,city,locality,city_tier,property_type,bhk,size_sqft,bathrooms,balconies,furnishing,floor,total_floors,age_of_property_years,has_parking,has_lift,has_security,has_gym,has_pool,has_power_backup,near_metro,distance_to_city_center_km,tenant_preference,available_from,security_deposit_inr,monthly_rent_inr
0,RNT100000,Bangalore,Marathahalli,1,PG/Co-living,2,244,2,0.0,Fully Furnished,2,25,21.0,0,1,1,0,0,0,1,8.7,Bachelors,2025-04-19,16400,8200
1,RNT100001,Hyderabad,Gachibowli,2,Apartment,2,866,1,1.0,Semi-Furnished,4,4,18.0,1,0,0,1,0,0,0,3.1,Family,2025-03-03,60900,20300
2,RNT100002,Pune,Hinjewadi,2,Apartment,3,1449,3,1.0,Semi-Furnished,30,30,17.0,1,1,1,1,0,0,0,4.1,Bachelors,2024-02-22,87200,43600
3,RNT100003,Hyderabad,Miyapur,2,PG/Co-living,1,252,1,0.0,Semi-Furnished,2,2,10.0,0,1,1,0,0,1,1,9.4,Any,2024-02-02,27000,4500
4,RNT100004,Mumbai,Thane,1,Apartment,2,1048,1,1.0,Unfurnished,2,15,0.0,1,1,0,0,0,0,0,2.8,Any,2024-04-05,257700,85900


In [0]:
Rental_prediction_pipeline.info()
Rental_prediction_pipeline.isnull().sum()
Rental_prediction_pipeline.shape


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   listing_id                  50000 non-null  object 
 1   city                        50000 non-null  object 
 2   locality                    50000 non-null  object 
 3   city_tier                   50000 non-null  int64  
 4   property_type               50000 non-null  object 
 5   bhk                         50000 non-null  int64  
 6   size_sqft                   50000 non-null  int64  
 7   bathrooms                   50000 non-null  int64  
 8   balconies                   50000 non-null  float64
 9   furnishing                  50000 non-null  object 
 10  floor                       50000 non-null  int64  
 11  total_floors                50000 non-null  int64  
 12  age_of_property_years       50000 non-null  float64
 13  has_parking                 500

(50000, 25)

## 3.Available Features

In [0]:
target = "monthly_rent_inr"

features = [
    "city",
    "city_tier",
    "property_type",
    "bhk",
    "size_sqft",
    "bathrooms",
    "balconies",
    "furnishing",
    "floor",
    "total_floors",
    "age_of_property_years",
    "has_parking",
    "has_lift",
    "has_security",
    "has_gym",
    "has_pool",
    "has_power_backup",
    "near_metro",
    "distance_to_city_center_km",
    "tenant_preference"
]

X = Rental_prediction_pipeline[features]
y = Rental_prediction_pipeline[target]

print(X.shape)
print(X.columns.tolist())

(50000, 20)
['city', 'city_tier', 'property_type', 'bhk', 'size_sqft', 'bathrooms', 'balconies', 'furnishing', 'floor', 'total_floors', 'age_of_property_years', 'has_parking', 'has_lift', 'has_security', 'has_gym', 'has_pool', 'has_power_backup', 'near_metro', 'distance_to_city_center_km', 'tenant_preference']


In [0]:
bool_cols=X.select_dtypes(include=bool).columns
X[bool_cols]=X[bool_cols].astype(int)



## 4.Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)


(40000, 20)
(10000, 20)


## 5.Split Numerical and Categorical Features

In [0]:
categorical_features = [
    "city",
    "property_type",
    "furnishing",
    "tenant_preference"
]



In [0]:
print("Categorical:", categorical_features)


Categorical: ['city', 'property_type', 'furnishing', 'tenant_preference']


## 5.Encoding Pipeline

In [0]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)


## 6.XGBoost Model


In [0]:
xgb=XGBRegressor(random_state=42, objective="reg:squarederror")

## 7. XGBoost Pipeline

In [0]:
xgb_pipeline=Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb)
])


## 8.Hyperparameter search space

In [0]:
param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

## 9.RandomizedSearchCV

In [0]:
random_search_xgb = RandomizedSearchCV( estimator=xgb_pipeline,param_distributions=param_dist,n_iter=20, cv=5, scoring='r2', random_state=42, n_jobs=-1
)

random_search_xgb.fit(X_train, y_train)

/databricks/python/lib/python3.12/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('categorical',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               ['city',
                                                                                'property_type',
                                                                                'furnishing',
                                                                                'tenant_preference'])])),
                                             ('model',
                                              XGBRegressor(base_score=None,
                                                           booster=None,
                                                           callbacks=None,
                                                           colsample_bylevel=None,
                                                           colsample_bynode=None,
                                                           c...
                                                           missing=nan,
                                                           monotone_constraints=None,
                                                           multi_strategy=None,
                                                           n_estimators=None,
                                                           n_jobs=None,
                                                           num_parallel_tree=None, ...))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__colsample_bytree': [0.8, 1.0],
                                        'model__learning_rate': [0.01, 0.05,
                                                                 0.1],
                                        'model__max_depth': [3, 5, 7],
                                        'model__n_estimators': [100, 200, 300],
                                        'model__subsample': [0.8, 1.0]},
                   random_state=42, scoring='r2')

## 10.Best Parameter

In [0]:
best_params =print("Best Parameters:", random_search_xgb.best_params_)
best_score =print("Best CV Score:", random_search_xgb.best_score_)

Best Parameters: {'model__subsample': 1.0, 'model__n_estimators': 200, 'model__max_depth': 5, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}
Best CV Score: 0.9825517535209656


## 11.Final Model

In [0]:
Final_model=random_search_xgb.best_estimator_
Final_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'property_type',
                                                   'furnishing',
                                                   'tenant_preference'])])),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=0.8, device=None,
                              ea...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=200, n_jobs=None,
                              num_parallel_tree=None, ...))])

## 12.Prediction

In [0]:
y_pred=Final_model.predict(X_test)


### Evaluation Metrics

In [0]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R2: 0.9826334118843079
MAE: 2969.81640625
RMSE: 5015.214252651625


## 13.Save final model

In [0]:
joblib.dump(Final_model, 'xgb_model.joblib')
print("Model saved Successfully")

Model saved Successfully
